In [1]:
# ============================================================
# CELL 1 — SETUP + LOAD STORED RAG DATA
# Uses existing test questions and retrievals from Drive
# ============================================================

!pip install -q "transformers>=4.48,<5" accelerate bitsandbytes \
    sacrebleu rapidfuzz bert-score==0.3.13 sentencepiece

from google.colab import drive
drive.mount("/content/drive")

import json, re, gc, unicodedata
import pandas as pd
import numpy as np
import torch
from pathlib import Path

# ---------- Find project folder ----------
possible = [
    Path("/content/drive/MyDrive/Govt_Chatbot"),
    Path("/content/drive/MyDrive/Govt_Chatbots")
]

BASE = None

for p in possible:
    if (p / "RAG" / "test_questions.csv").exists():
        BASE = p
        break

if BASE is None:
    raise FileNotFoundError("Govt_Chatbot/RAG folder not found.")

RAG_DIR = BASE / "RAG"
OUT = BASE / "Mistral" / "RAG"

OUT.mkdir(parents=True, exist_ok=True)

# ---------- Load test questions ----------
tests = pd.read_csv(
    RAG_DIR / "test_questions.csv"
).fillna("")

# ---------- Load already stored retrievals ----------
with open(
    RAG_DIR / "retrievals.json",
    encoding="utf-8"
) as f:
    retrievals = json.load(f)

assert len(tests) == len(retrievals), \
    "Test questions and retrievals count mismatch."

assert "question" in tests.columns
assert "gold" in tests.columns

# Save exact inputs used
tests.to_csv(
    OUT / "test_questions_used.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    OUT / "retrievals_used.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        retrievals,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Test questions:", len(tests))
print("Retrieval sets:", len(retrievals))
print("Output folder:", OUT)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 12.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatib

In [2]:
# ============================================================
# CELL 2 — MISTRAL-7B RAG INFERENCE
# Stored Top-5 contexts -> Mistral-7B-Instruct-v0.3
# max_new_tokens = 500
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from tqdm.auto import tqdm

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_NEW_TOKENS = 500

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU runtime first.")

# ---------- Load model in 4-bit ----------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Loading Mistral-7B-Instruct-v0.3...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

device = model.get_input_embeddings().weight.device


# ============================================================
# GENERATION FUNCTION
# ============================================================

SYSTEM_PROMPT = (
    "তুমি বাংলাদেশের সরকারি সেবা বিষয়ক একজন সহকারী। "
    "শুধুমাত্র প্রদত্ত Context ব্যবহার করে প্রশ্নের সঠিক ও সংক্ষিপ্ত উত্তর বাংলায় দাও। "
    "প্রশ্ন ও Context-এর ভাষা এক না হলেও সমার্থক তথ্য বুঝে উত্তর দাও। "
    "ফি, সময়, সংখ্যা, প্রয়োজনীয় কাগজপত্র ও প্রক্রিয়া নির্ভুলভাবে উল্লেখ করো। "
    "অপ্রয়োজনীয় ব্যাখ্যা দিও না। "
    "Context-এ উত্তর একেবারেই না থাকলে বলবে: "
    "'প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।'"
)


def generate_answer(question, docs):

    context = "\n\n".join([
        f"[Context {i}]\n"
        f"শিরোনাম: {doc.get('title', '')}\n"
        f"তথ্য: {doc.get('text', '')}"
        for i, doc in enumerate(docs, 1)
    ])

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content":
                f"Context:\n{context}\n\n"
                f"প্রশ্ন: {question}\n"
                f"উত্তর:"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7000
    ).to(device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and generated[-1].item() != tokenizer.eos_token_id
    )

    return answer, truncated


# ============================================================
# RESUME SUPPORT
# ============================================================

PARTIAL = OUT / "predictions_partial.csv"

done = {}

if PARTIAL.exists():

    old = pd.read_csv(PARTIAL).fillna("")

    for _, row in old.iterrows():

        done[
            (
                str(row["domain"]),
                str(row["id"])
            )
        ] = row.to_dict()

print("Already completed:", len(done))


# ============================================================
# INFERENCE
# ============================================================

predictions = []

for i, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Mistral RAG"
):

    key = (
        str(row["domain"]),
        str(row["id"])
    )

    if key in done:

        result = done[key]

    else:

        answer, truncated = generate_answer(
            row["question"],
            retrievals[i]
        )

        result = {
            "id": str(row["id"]),
            "domain": str(row["domain"]),
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "truncated": truncated
        }

        done[key] = result

    predictions.append(result)

    # checkpoint after every answer
    pd.DataFrame(predictions).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig"
    )


# ---------- Save final predictions ----------
pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

truncated_count = (
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
    .sum()
)

print("\nCompleted:", len(pred_df))
print("Truncated outputs:", truncated_count)
print("Saved:", OUT / "predictions.csv")


# Free GPU before BERTScore
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Already completed: 0


Mistral RAG:   0%|          | 0/248 [00:00<?, ?it/s]


Completed: 248
Truncated outputs: 42
Saved: /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG/predictions.csv


In [3]:
# ============================================================
# CELL 3 — EVALUATION
# Exact Match, Fuzzy Match, Corpus BLEU,
# ROUGE-1/2/L, Token F1,
# BERT Precision/Recall/F1 + Truncated Outputs
# ============================================================

from collections import Counter
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score

df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (
    df["gold"]
    .astype(str)
    .str.strip()
    != ""
).all(), "Blank gold answer found."


# ---------- Normalization ----------
BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(BN_TO_EN).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ---------- Exact Match ----------
def exact_match(pred, gold):

    return float(
        normalize(pred) == normalize(gold)
    )


# ---------- Token F1 ----------
def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (Counter(p) & Counter(g)).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ---------- ROUGE-N ----------
def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pn & gn).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ---------- ROUGE-L ----------
def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)

            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ============================================================
# ROW-LEVEL METRICS
# ============================================================

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(df["prediction"], df["gold"])
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu]
    ).score / 100
)


# ============================================================
# BERTSCORE
# ============================================================

print("Calculating BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=4,
    device="cpu",
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ---------- Truncation ----------
truncated_count = (
    df["truncated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
    .sum()
)


# ============================================================
# FINAL RESULT
# ============================================================

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1",
        "Truncated Outputs"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean(),
        int(truncated_count)
    ]
})


# Save row-level metrics
df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save overall result
result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved:")
print(OUT / "test_questions_used.csv")
print(OUT / "retrievals_used.json")
print(OUT / "predictions_partial.csv")
print(OUT / "predictions.csv")
print(OUT / "result.csv")

Calculating BERTScore...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 53.66 seconds, 4.62 sentences/sec


,metric,score
0,Exact Match,0.008065
1,Fuzzy Match,0.762482
2,Corpus BLEU,0.220559
3,ROUGE-1,0.453566
4,ROUGE-2,0.351079
5,ROUGE-L,0.422116
6,Token F1,0.453566
7,BERT Precision,0.768958
8,BERT Recall,0.813552
9,BERT F1,0.788607



Saved:
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG/test_questions_used.csv
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG/retrievals_used.json
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG/predictions_partial.csv
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG/predictions.csv
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG/result.csv
